In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog", "de_dev")
catalog = dbutils.widgets.get("catalog")
print(catalog)


In [0]:
df = spark.table(f"{catalog}.bronze.suppliers")
clean_df = (
    df
    .dropna(subset=["supplier_id" ,"supplier_name","contact_email"])
    .dropDuplicates(["supplier_id"])
    .fillna({"country":"Unknown"})
    .select(
        "supplier_id" ,"supplier_name","contact_email","country"
    )
)
clean_df.createOrReplaceTempView("suppliers_clean_view")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.silver.suppliers_scd_1 (
    supplier_id int,
    supplier_name string,
    contact_email string,
    country string
)
USING DELTA;

In [0]:
%sql
MERGE INTO ${catalog}.silver.suppliers_scd_1 AS trg
USING suppliers_clean_view AS src
ON trg.supplier_id = src.supplier_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *